# 02 — Data Cleaning

**Thesis:** Comparative Analysis of Machine Learning Algorithms for Predicting CVD Risk in a Tunisian Hospital Population

## Purpose of this notebook
This notebook performs the full data-cleaning stage of the pipeline, between raw data collection and exploratory data analysis (EDA). It transforms the raw, unprocessed clinical dataset into a well-documented, analysis-ready dataset while preserving every original record for traceability.

## Dataset source
Raw file: `data/raw/CVD_not_clean.csv`

## Objective of data cleaning
1. Assess data quality (missing values, duplicates, inconsistent categories, implausible numerical values).
2. Standardize and numerically encode categorical variables, replacing (not duplicating) the original columns.
3. Validate clinically-derived variables (BMI, Blood Pressure) for internal consistency.
4. Treat missing values using a justified, documented strategy.
5. Produce a single, clean, ML-ready CSV file, together with a transparent before/after audit trail.

## Difference between "raw" and "cleaned" data
- Raw data (`CVD_not_clean.csv`) is the original export exactly as received. It is never modified or overwritten by this notebook.
- Cleaned data (`CVD_cleaned.csv`) is a new, derived file: standardized categories, numerically encoded variables, corrected/validated clinical fields, and imputed missing values, ready to be consumed by the EDA notebook and, later, by the ML modeling notebooks.

> **Note on data leakage:** Some preprocessing steps (e.g., missing-value imputation, scaling) are performed here at the *dataset* level to produce a coherent, explorable dataset for the thesis. When the ML models are trained later, these steps must be **refit inside the cross-validation/train-test pipeline** (e.g., using `sklearn.pipeline.Pipeline` with `SimpleImputer` fit only on the training fold) so that no information from the test set leaks into training. This distinction is flagged explicitly at each relevant step below.

## 0. Setup: Imports and Paths

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Project paths (relative -- portable) ---
PROJECT_DIR = Path.cwd().parent
RAW_CSV_PATH = PROJECT_DIR / "data" / "raw" / "CVD_not_clean.csv"

CLEANED_DIR = PROJECT_DIR / "data" / "cleaned"
CLEANED_CSV_PATH = CLEANED_DIR / "CVD_cleaned.csv"

FIGURES_DIR = PROJECT_DIR / "figures"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 100,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})
sns.set_style("whitegrid")

print("Raw file      :", RAW_CSV_PATH)
print("Cleaned output:", CLEANED_CSV_PATH)
print("Figures folder:", FIGURES_DIR)


Raw file      : /home/claude/ML_HOSPITAL/data/raw/CVD_not_clean.csv
Cleaned output: /home/claude/ML_HOSPITAL/data/cleaned/CVD_cleaned.csv
Figures folder: /home/claude/ML_HOSPITAL/figures


In [2]:
def save_fig(fig, filename):
    # Save a figure into the figures folder with consistent settings, then close it.
    path = FIGURES_DIR / filename
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Figure saved -> {path}")
    plt.close(fig)


## 1. Load Raw Dataset

In [3]:
df_raw = pd.read_csv(RAW_CSV_PATH, sep=";")
df = df_raw.copy()  # work on a copy; df_raw is kept pristine for reference

print("Shape (rows, columns):", df_raw.shape)
df_raw.head()


Shape (rows, columns): (1529, 15)


,Sex,Age,Weight (kg),Height (cm),BMI,Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),Smoking Status,Diabetes Status,Physical Activity Level,Family History of CVD,Systolic BP,Diastolic BP,CVD Risk Level
0,F,32.0,69.1,171.0,23.6,248.0,78.0,111.0,N,Y,Low,N,125.0,79.0,INTERMEDIARY
1,F,55.0,118.7,169.0,41.6,162.0,50.0,135.0,Y,Y,High,Y,139.0,70.0,HIGH
2,M,NaN,NaN,183.0,26.9,103.0,73.0,114.0,N,N,High,Y,104.0,77.0,INTERMEDIARY
3,M,44.0,108.3,NaN,33.4,134.0,46.0,91.0,N,N,High,Y,140.0,83.0,INTERMEDIARY
4,F,32.0,99.5,186.0,28.8,146.0,64.0,141.0,Y,Y,High,N,144.0,83.0,INTERMEDIARY


In [4]:
assert RAW_CSV_PATH.exists(), "Raw file missing - check the path."
pd.testing.assert_frame_equal(df_raw, pd.read_csv(RAW_CSV_PATH, sep=";"))
print("Confirmed: df_raw is identical to the file on disk. The raw dataset has not been modified.")


Confirmed: df_raw is identical to the file on disk. The raw dataset has not been modified.


## 2. Data Quality Assessment

In [5]:
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
}).sort_values("missing_count", ascending=False)

print("Missing values per column:\n")
missing_report


Missing values per column:



,missing_count,missing_pct
Diastolic BP,82,5.36
Weight (kg),81,5.30
HDL (mg/dL),80,5.23
Age,78,5.10
Height (cm),74,4.84
Total Cholesterol (mg/dL),73,4.77
Systolic BP,71,4.64
Fasting Blood Sugar (mg/dL),67,4.38
BMI,64,4.19
Sex,0,0.00


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = missing_report[missing_report["missing_count"] > 0].sort_values("missing_pct")
ax.barh(plot_data.index, plot_data["missing_pct"], color="#c0392b")
ax.set_xlabel("Missing values (%)")
ax.set_title("Missing Values per Column -- Raw Dataset")
for y, v in enumerate(plot_data["missing_pct"]):
    ax.text(v + 0.05, y, f"{v}%", va="center", fontsize=9)
save_fig(fig, "01_missing_values_before.png")


Figure saved -> /home/claude/ML_HOSPITAL/figures/01_missing_values_before.png


In [7]:
n_duplicates_raw = df.duplicated().sum()
print(f"Exact duplicate rows in raw data: {n_duplicates_raw}")


Exact duplicate rows in raw data: 0


In [8]:
categorical_cols = [
    "Sex", "Smoking Status", "Diabetes Status",
    "Physical Activity Level", "Family History of CVD", "CVD Risk Level"
]
for col in categorical_cols:
    print(f"{col:30s} -> {sorted(df[col].dropna().unique().tolist())}")


Sex                            -> ['F', 'M']
Smoking Status                 -> ['N', 'Y']
Diabetes Status                -> ['N', 'Y']
Physical Activity Level        -> ['High', 'Low', 'Moderate']
Family History of CVD          -> ['N', 'Y']
CVD Risk Level                 -> ['HIGH', 'INTERMEDIARY', 'LOW']


In [9]:
numeric_cols = [
    "Age", "Weight (kg)", "Height (cm)", "BMI",
    "Total Cholesterol (mg/dL)", "HDL (mg/dL)", "Fasting Blood Sugar (mg/dL)",
    "Systolic BP", "Diastolic BP"
]
df[numeric_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
Age,1451.0,47.025500,12.421063,25.0,37.000,46.0000,55.0000,79.00
Weight (kg),1448.0,85.917427,21.012580,50.1,67.050,86.6145,105.0000,120.00
Height (cm),1455.0,175.390600,11.251527,150.0,166.000,175.6940,184.2085,199.96
BMI,1465.0,28.465997,7.038685,15.0,22.629,28.1590,34.0000,46.20
Total Cholesterol (mg/dL),1456.0,198.539148,57.794099,100.0,150.000,197.0000,249.0000,300.00
HDL (mg/dL),1449.0,56.197378,16.066754,30.0,42.000,56.0000,70.0000,89.00
Fasting Blood Sugar (mg/dL),1462.0,117.485636,30.289174,70.0,92.000,115.0000,138.0000,198.00
Systolic BP,1458.0,125.627572,22.112099,90.0,107.000,125.0000,141.0000,179.00
Diastolic BP,1447.0,82.917761,14.731277,60.0,71.000,82.0000,93.0000,119.00


## 3. Missing Values

**Chosen treatment: median imputation.**

For each of the nine numerical variables, missing counts are all below ~5.5% of the sample -- low enough that deletion would discard usable information from otherwise complete records, but present in numerous rows, so simply dropping them is not appropriate. Clinical variables such as BMI, cholesterol, or blood pressure are typically right-skewed and can contain legitimate extreme values; the median is more robust to that skew than the mean.

**Important -- sequencing decision.** BMI and Blood Pressure undergo additional validity checks later (Sections 5-6) which can themselves introduce a small number of new missing values. To avoid imputing twice, the actual imputation is executed once, at the end of Section 6, after all validity corrections have been applied.

**Data-leakage note.** The medians used here are computed once from the full dataset, purely to produce a coherent, explorable dataset for the thesis (data cleaning + EDA). This is not appropriate to reuse when training/evaluating ML models later: at the modeling stage, imputation must be refit inside each train/validation fold.

## 4. Categorical Standardization and Encoding

| Original variable | Encoding rule |
|---|---|
| Sex | F -> 0, M -> 1 |
| Smoking Status | N -> 0, Y -> 1 |
| Diabetes Status | N -> 0, Y -> 1 |
| Family History of CVD | N -> 0, Y -> 1 |
| Physical Activity Level | Low -> 0, Moderate -> 1, High -> 2 (ordinal) |
| CVD Risk Level (target) | LOW -> 0, INTERMEDIARY -> 1, HIGH -> 2 (ordinal) |

In [10]:
binary_yn_cols = ["Smoking Status", "Diabetes Status", "Family History of CVD"]
yn_map = {"N": 0, "Y": 1}
for col in binary_yn_cols:
    df[col] = df[col].map(yn_map).astype("Int64")

sex_map = {"F": 0, "M": 1}
df["Sex"] = df["Sex"].map(sex_map).astype("Int64")

activity_map = {"Low": 0, "Moderate": 1, "High": 2}
df["Physical Activity Level"] = df["Physical Activity Level"].map(activity_map).astype("Int64")

risk_map = {"LOW": 0, "INTERMEDIARY": 1, "HIGH": 2}
df["CVD Risk Level"] = df["CVD Risk Level"].map(risk_map).astype("Int64")

encoded_cols = binary_yn_cols + ["Sex", "Physical Activity Level", "CVD Risk Level"]
for col in encoded_cols:
    assert df[col].isna().sum() == 0, f"Unmapped category found in {col} - check for typos."

print("Encoding applied. Post-encoding value counts:\n")
for col in encoded_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).sort_index())
    print()


Encoding applied. Post-encoding value counts:

--- Smoking Status ---
Smoking Status
0    740
1    789
Name: count, dtype: Int64

--- Diabetes Status ---
Diabetes Status
0    752
1    777
Name: count, dtype: Int64

--- Family History of CVD ---
Family History of CVD
0    780
1    749
Name: count, dtype: Int64

--- Sex ---
Sex
0    773
1    756
Name: count, dtype: Int64

--- Physical Activity Level ---
Physical Activity Level
0    496
1    512
2    521
Name: count, dtype: Int64

--- CVD Risk Level ---
CVD Risk Level
0    220
1    581
2    728
Name: count, dtype: Int64



## 5. Numerical Validation

In [11]:
plausible_ranges = {
    "Age": (0, 120),
    "Weight (kg)": (20, 300),
    "Height (cm)": (100, 250),
    "BMI": (10, 80),
    "Total Cholesterol (mg/dL)": (50, 500),
    "HDL (mg/dL)": (10, 150),
    "Fasting Blood Sugar (mg/dL)": (30, 400),
    "Systolic BP": (60, 260),
    "Diastolic BP": (30, 160),
}

validation_rows = []
for col, (low, high) in plausible_ranges.items():
    out_of_range = df[(df[col] < low) | (df[col] > high)]
    validation_rows.append({
        "variable": col,
        "plausible_min": low,
        "plausible_max": high,
        "observed_min": df[col].min(),
        "observed_max": df[col].max(),
        "n_out_of_range": len(out_of_range),
    })

numeric_validation_report = pd.DataFrame(validation_rows)
numeric_validation_report


,variable,plausible_min,plausible_max,observed_min,observed_max,n_out_of_range
0,Age,0,120,25.0,79.00,0
1,Weight (kg),20,300,50.1,120.00,0
2,Height (cm),100,250,150.0,199.96,0
3,BMI,10,80,15.0,46.20,0
4,Total Cholesterol (mg/dL),50,500,100.0,300.00,0
5,HDL (mg/dL),10,150,30.0,89.00,0
6,Fasting Blood Sugar (mg/dL),30,400,70.0,198.00,0
7,Systolic BP,60,260,90.0,179.00,0
8,Diastolic BP,30,160,60.0,119.00,0


In [12]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color="#3498db")
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("")
fig.suptitle("Distribution of Numerical Variables -- Before Correction (Boxplots)")
fig.tight_layout()
save_fig(fig, "03_outliers_before.png")


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_outliers_before.png


## 6. BMI Validation

Body Mass Index has a well-defined clinical formula: BMI = Weight (kg) / Height (m)^2

We recompute BMI from the recorded Weight and Height for every row where both are available, and compare it to the recorded BMI value.

In [13]:
height_m = df["Height (cm)"] / 100
bmi_calculated = df["Weight (kg)"] / (height_m ** 2)
bmi_diff = (bmi_calculated - df["BMI"]).abs()
can_compare = df["Weight (kg)"].notna() & df["Height (cm)"].notna() & df["BMI"].notna()

print(f"Rows where BMI can be cross-checked: {can_compare.sum()} / {len(df)}")
print(f"Rows with |recorded BMI - calculated BMI| > 1.0 kg/m^2: {(bmi_diff[can_compare] > 1.0).sum()}")
print()
print("Discrepancy statistics (kg/m^2), among comparable rows:")
bmi_diff[can_compare].describe()


Rows where BMI can be cross-checked: 1317 / 1529
Rows with |recorded BMI - calculated BMI| > 1.0 kg/m^2: 431

Discrepancy statistics (kg/m^2), among comparable rows:


count    1317.000000
mean        3.022872
std         5.508018
min         0.000051
25%         0.020305
50%         0.040025
75%         3.968987
max        31.421891
dtype: float64

**Decision:** BMI is a *deterministic function* of Weight and Height, not an independently measured quantity, so when the recorded BMI disagrees with the value calculated from the same patient's recorded Weight and Height, the calculated value is treated as the more reliable one.

**Rule applied:**
- If Weight and Height are both available: replace `BMI` with the recalculated value (rounded to 1 decimal place), regardless of the size of the discrepancy.
- If Weight or Height is missing: the recorded `BMI` value (if present) is left untouched at this stage; it will be handled by median imputation only if it is itself missing.

In [14]:
rows_recalculated = df["Weight (kg)"].notna() & df["Height (cm)"].notna()
df.loc[rows_recalculated, "BMI"] = (
    df.loc[rows_recalculated, "Weight (kg)"] /
    ((df.loc[rows_recalculated, "Height (cm)"] / 100) ** 2)
).round(1)

print(f"BMI recalculated and replaced for {rows_recalculated.sum()} rows "
      f"(all rows with non-missing Weight and Height).")


BMI recalculated and replaced for 1375 rows (all rows with non-missing Weight and Height).


## 7. Blood Pressure Validation

Physiologically, Systolic BP must be strictly greater than Diastolic BP. We check every row for this constraint.

In [15]:
bp_present = df["Systolic BP"].notna() & df["Diastolic BP"].notna()
impossible_bp = bp_present & (df["Systolic BP"] <= df["Diastolic BP"])

print(f"Rows with Systolic BP <= Diastolic BP: {impossible_bp.sum()} out of {bp_present.sum()} rows with both values present")
df.loc[impossible_bp, ["Systolic BP", "Diastolic BP"]].describe()


Rows with Systolic BP <= Diastolic BP: 65 out of 1378 rows with both values present


,Systolic BP,Diastolic BP
count,65.000000,65.000000
mean,98.584615,105.876923
std,7.828019,8.814951
min,90.000000,91.000000
25%,92.000000,98.000000
50%,95.000000,107.000000
75%,105.000000,114.000000
max,118.000000,119.000000


In [16]:
swap_would_fix = impossible_bp & (df["Diastolic BP"] > df["Systolic BP"])
still_equal_after_swap = impossible_bp & (df["Diastolic BP"] == df["Systolic BP"])

print(f"Rows where Systolic and Diastolic appear to be swapped: {swap_would_fix.sum()}")
print(f"Rows where Systolic == Diastolic (swapping does NOT fix it): {still_equal_after_swap.sum()}")


Rows where Systolic and Diastolic appear to be swapped: 59
Rows where Systolic == Diastolic (swapping does NOT fix it): 6


**Decision, documented case by case:**
1. **Swapped values:** both values are individually plausible and simply swapping produces a fully consistent reading. Decision: swap the Systolic and Diastolic values for these rows.
2. **Equal values:** not physiologically valid, and swapping cannot resolve it. Decision: set both values to missing (`NaN`) so they are handled by the same, transparent median-imputation step as any other missing BP reading.

In [17]:
df.loc[swap_would_fix, ["Systolic BP", "Diastolic BP"]] = (
    df.loc[swap_would_fix, ["Diastolic BP", "Systolic BP"]].values
)

df.loc[still_equal_after_swap, ["Systolic BP", "Diastolic BP"]] = np.nan

bp_present_after = df["Systolic BP"].notna() & df["Diastolic BP"].notna()
remaining_impossible = bp_present_after & (df["Systolic BP"] <= df["Diastolic BP"])
print(f"Remaining Systolic <= Diastolic cases after correction: {remaining_impossible.sum()}")


Remaining Systolic <= Diastolic cases after correction: 0


### Executing the missing-value imputation (deferred from Section 3)

All validity corrections that could introduce new missing values (BMI, Blood Pressure) have now been applied. We perform the median imputation for all nine numerical variables once, here.

In [18]:
missing_before_impute = df[numeric_cols].isna().sum()
medians = {}
for col in numeric_cols:
    median_value = df[col].median()
    medians[col] = median_value
    df[col] = df[col].fillna(median_value)

missing_after_impute = df[numeric_cols].isna().sum()

impute_report = pd.DataFrame({
    "median_used": pd.Series(medians),
    "missing_before": missing_before_impute,
    "missing_after": missing_after_impute,
})
impute_report


,median_used,missing_before,missing_after
Age,46.0000,78,0
Weight (kg),86.6145,81,0
Height (cm),175.6940,74,0
BMI,27.7000,6,0
Total Cholesterol (mg/dL),197.0000,73,0
HDL (mg/dL),56.0000,80,0
Fasting Blood Sugar (mg/dL),115.0000,67,0
Systolic BP,125.0000,77,0
Diastolic BP,82.0000,88,0


In [19]:
fig, ax = plt.subplots(figsize=(8, 3))
after = df.isna().mean() * 100
ax.barh(after.index, after.values, color="#27ae60")
ax.set_xlabel("Missing values (%)")
ax.set_xlim(0, max(5, missing_report['missing_pct'].max()))
ax.set_title("Missing Values per Column -- After Cleaning")
save_fig(fig, "02_missing_values_after.png")


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_missing_values_after.png


## 8. Duplicate Handling

In [20]:
rows_before_dedup = len(df)
n_duplicates = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
rows_after_dedup = len(df)

print(f"Before : {rows_before_dedup} rows")
print(f"Removed: {n_duplicates} exact duplicate rows")
print(f"After  : {rows_after_dedup} rows")


Before : 1529 rows
Removed: 0 exact duplicate rows
After  : 1529 rows


## 9. Final Data Validation

In [21]:
final_checks = {
    "Missing values (total)": int(df.isna().sum().sum()),
    "Duplicate rows": int(df.duplicated().sum()),
    "Sex values": sorted(df["Sex"].unique().tolist()),
    "Smoking Status values": sorted(df["Smoking Status"].unique().tolist()),
    "Diabetes Status values": sorted(df["Diabetes Status"].unique().tolist()),
    "Physical Activity Level values": sorted(df["Physical Activity Level"].unique().tolist()),
    "Family History of CVD values": sorted(df["Family History of CVD"].unique().tolist()),
    "CVD Risk Level values": sorted(df["CVD Risk Level"].unique().tolist()),
    "Systolic <= Diastolic remaining": int((df["Systolic BP"] <= df["Diastolic BP"]).sum()),
    "Rows": df.shape[0],
    "Columns": df.shape[1],
}
for k, v in final_checks.items():
    print(f"{k:40s}: {v}")


Missing values (total)                  : 0
Duplicate rows                          : 0
Sex values                              : [0, 1]
Smoking Status values                   : [0, 1]
Diabetes Status values                  : [0, 1]
Physical Activity Level values          : [0, 1, 2]
Family History of CVD values            : [0, 1]
CVD Risk Level values                   : [0, 1, 2]
Systolic <= Diastolic remaining         : 0
Rows                                    : 1529
Columns                                 : 15


In [22]:
print("Final dtypes:\n")
print(df.dtypes)


Final dtypes:

Sex                              Int64
Age                            float64
Weight (kg)                    float64
Height (cm)                    float64
BMI                            float64
Total Cholesterol (mg/dL)      float64
HDL (mg/dL)                    float64
Fasting Blood Sugar (mg/dL)    float64
Smoking Status                   Int64
Diabetes Status                  Int64
Physical Activity Level          Int64
Family History of CVD            Int64
Systolic BP                    float64
Diastolic BP                   float64
CVD Risk Level                   Int64
dtype: object


## 10. Before vs. After Summary

In [23]:
invalid_before = (
    missing_report["missing_count"].sum()
    + int((bmi_diff[can_compare] > 1.0).sum())
    + int(impossible_bp.sum())
)
invalid_after = int(df.isna().sum().sum()) + int((df["Systolic BP"] <= df["Diastolic BP"]).sum())

before_after = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing values", "Duplicate rows", "Invalid values (BMI/BP inconsistencies + missing)"],
    "Before": [df_raw.shape[0], df_raw.shape[1], int(missing_report["missing_count"].sum()), 0, invalid_before],
    "After":  [df.shape[0], df.shape[1], int(df.isna().sum().sum()), int(n_duplicates), invalid_after],
})
before_after


,Metric,Before,After
0,Rows,1529,1529
1,Columns,15,15
2,Missing values,670,0
3,Duplicate rows,0,0
4,Invalid values (BMI/BP inconsistencies + missing),1166,0


## 11. Export Cleaned Dataset

Only the final, cleaned dataset is exported. It contains exactly the original 15 variables -- no duplicate or intermediate columns -- with categorical variables numerically encoded in place and all clinical fields validated.

In [24]:
assert df.shape[1] == 15, "Unexpected number of columns - review the pipeline before export."
assert df.isna().sum().sum() == 0, "Unexpected missing values remain - review before export."

df.to_csv(CLEANED_CSV_PATH, index=False, sep=';')

print("Export complete.")
print("File exists:", CLEANED_CSV_PATH.exists())
print("Final shape:", df.shape)
print("Final columns:", df.columns.tolist())


Export complete.
File exists: True
Final shape: (1529, 15)
Final columns: ['Sex', 'Age', 'Weight (kg)', 'Height (cm)', 'BMI', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Systolic BP', 'Diastolic BP', 'CVD Risk Level']


In [25]:
df.head()

,Sex,Age,Weight (kg),Height (cm),BMI,Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),Smoking Status,Diabetes Status,Physical Activity Level,Family History of CVD,Systolic BP,Diastolic BP,CVD Risk Level
0,0,32.0,69.1000,171.000,23.6,248.0,78.0,111.0,0,1,0,0,125.0,79.0,1
1,0,55.0,118.7000,169.000,41.6,162.0,50.0,135.0,1,1,2,1,139.0,70.0,2
2,1,46.0,86.6145,183.000,26.9,103.0,73.0,114.0,0,0,2,1,104.0,77.0,1
3,1,44.0,108.3000,175.694,33.4,134.0,46.0,91.0,0,0,2,1,140.0,83.0,1
4,0,32.0,99.5000,186.000,28.8,146.0,64.0,141.0,1,1,2,0,144.0,83.0,1


## What was done
- Loaded the raw dataset (1,529 rows x 15 columns) without modifying the source file.
- Assessed data quality: confirmed consistent categorical spelling, no exact duplicates, and no single-variable impossible values, while identifying ~4-5.5% missingness across all nine numerical variables.
- Standardized and numerically encoded six categorical variables **in place**, with no duplicate columns.
- Validated BMI against the clinical formula and recalculated it for every row where Weight and Height were both available.
- Validated Blood Pressure logic: corrected apparently transposed readings by swapping, and set unresolvable equal-value pairs to missing.
- Imputed all remaining numerical missing values using dataset-level medians, explicitly flagging that this must be redone inside the ML training pipeline to avoid data leakage.
- Verified duplicates and re-ran a full final validation confirming zero missing values, zero duplicates, and zero BP inconsistencies.
- Exported the cleaned dataset to `data/cleaned/CVD_cleaned.csv`.

## What comes next
`03_EDA.ipynb` will load only `CVD_cleaned.csv` and perform exploratory data analysis.